# RAG Sprint 1 – מנוע חיפוש סמנטי על "מדריך לרוכש דירה"

נוטבוק זה בונה את שכבת ה־**Retrieval** של מערכת RAG, שלב אחר שלב:

1. **טעינת המסמך** – קריאת ה־PDF והמרתו לטקסט.
2. **חלוקה ל־Chunks** – פירוק הטקסט לקטעים קטנים וחופפים.
3. **Embeddings** – המרת כל chunk לוקטור מספרים באמצעות `gemini-embedding-001`.
4. **Pinecone** – שמירת הוקטורים במסד נתונים וקטורי.
5. **Semantic Search** – חיפוש לפי משמעות + הצגת התוצאות הרלוונטיות.

> כל שלב מתועד בתא Markdown שמסביר מה קורה ולמה.

## שלב 2 – טעינת המסמך וחלוקה ל־Chunks

**למה מחלקים ל־Chunks?**
מודל ה־Embeddings וה־Retrieval עובדים טוב יותר על קטעים קצרים וממוקדים מאשר על מסמך שלם.
בנוסף, יש הגבלת אורך לקלט. לכן מפרקים את הטקסט לקטעים ("chunks").

**מה זה `chunk_overlap`?**
כדי לא "לחתוך" משפט או רעיון באמצע בין שני chunks, אנחנו משאירים חפיפה של כמה תווים
בין chunk לחבירו. כך מידע שנמצא על הגבול עדיין מופיע בשלמותו באחד הקטעים.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pypdf import PdfReader

# --- טעינת מפתחות ה-API מקובץ .env ---
# load_dotenv מחפש את הקובץ .env ומכניס את המשתנים ל-os.environ.
# find .env בתיקיית השורש של הפרויקט (רמה אחת מעל notebooks/).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# בדיקה שהמפתחות נטענו (בלי להדפיס אותם!)
print("GEMINI_API_KEY נטען:", "כן" if GEMINI_API_KEY else "לא - בדקי את קובץ .env")
print("PINECONE_API_KEY נטען:", "כן" if PINECONE_API_KEY else "לא - בדקי את קובץ .env")

# --- הגדרות מרכזיות (נשנה אותן בשלב הבונוס) ---
PDF_PATH = PROJECT_ROOT / "data" / "apartment_buyer_guide.pdf"
CHUNK_SIZE = 1000       # אורך כל chunk בתווים
CHUNK_OVERLAP = 150     # חפיפה בתווים בין chunks עוקבים

EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768         # מימד הוקטור (חייב להתאים למימד ה-index ב-Pinecone)
INDEX_NAME = "rag-apartment-guide"

print("\nמסמך:", PDF_PATH)
print("קיים:", PDF_PATH.exists())

In [ ]:
import re


def clean_text(text):
    """ניקוי טקסט שחולץ מ-PDF: מכווץ רצפים ארוכים של רווחים ושורות ריקות
    (ה-PDF מכיל הרבה שורות ריקות שמוסיפות רעש ל-Embeddings)."""
    text = re.sub(r"[ \t]+", " ", text)      # רצף רווחים -> רווח בודד
    text = re.sub(r"\n\s*\n+", "\n", text)   # שורות ריקות מרובות -> שורה אחת
    return text.strip()


def load_pdf(pdf_path):
    """קורא PDF ומחזיר:
    - full_text: כל הטקסט של המסמך כמחרוזת אחת (מנוקה)
    - char_page: רשימה שממפה כל תו במיקום i למספר העמוד שממנו הגיע (לצורך metadata)
    """
    reader = PdfReader(str(pdf_path))
    full_text = ""
    char_page = []
    for page_num, page in enumerate(reader.pages, start=1):
        page_text = clean_text(page.extract_text() or "") + "\n"
        full_text += page_text
        char_page.extend([page_num] * len(page_text))
    return full_text, char_page


full_text, char_page = load_pdf(PDF_PATH)

print(f"מספר עמודים: {char_page[-1] if char_page else 0}")
print(f"סך תווים בטקסט: {len(full_text):,}")
print("\n--- תצוגה מקדימה (300 תווים ראשונים) ---")
print(full_text[:300])

In [ ]:
def chunk_text(text, char_page, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """מחלק טקסט ארוך לקטעים בגודל chunk_size תווים, עם חפיפה של chunk_overlap.
    כל chunk מקבל metadata עם מספר העמוד שבו הוא מתחיל.
    מחזיר רשימת dict: {id, text, page}.
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap חייב להיות קטן מ-chunk_size")

    chunks = []
    step = chunk_size - chunk_overlap  # כמה מתקדמים כל פעם
    idx = 0
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_str = text[start:end].strip()
        if chunk_str:  # מדלגים על קטעים ריקים
            page = char_page[start] if start < len(char_page) else char_page[-1]
            chunks.append({
                "id": f"chunk-{idx}",
                "text": chunk_str,
                "page": page,
            })
            idx += 1
        start += step
    return chunks

In [ ]:
chunks = chunk_text(full_text, char_page, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"נוצרו {len(chunks)} chunks (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
print("\n--- דוגמה: ה-chunk הראשון ---")
print("id:", chunks[0]["id"], "| עמוד:", chunks[0]["page"])
print(chunks[0]["text"][:400], "...")

print("\n--- דוגמה: chunk מאמצע המסמך ---")
mid = len(chunks) // 2
print("id:", chunks[mid]["id"], "| עמוד:", chunks[mid]["page"])
print(chunks[mid]["text"][:400], "...")